In [2]:
!pip install langchain langchain-community langchain-huggingface sentence-transformers chromadb


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
!pip install langchain-text-splitters langchain_core


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from langchain_core.documents import Document

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [7]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [8]:
!pip install -qU "langchain-chroma>=0.1.2"


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6866.07it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
from langchain_chroma import Chroma

vector_store = Chroma(
    embedding_function= embeddings,
    persist_directory="chroma_db",
    collection_name="ipl_players"
)

In [15]:
vector_store.add_documents(docs)

['8889fe21-079e-48cf-b2fe-cce2c44d36a4',
 'e931680c-d6a5-417a-a43e-8ebc8da419c1',
 '2c2dc62a-6d11-4608-bbf6-b458bedf6110',
 '3c64f402-1e59-45da-b265-da5cc47406a2',
 '71881230-f996-42bd-9142-207058c9bf1a']

In [16]:
vector_store.get(include=["embeddings", "metadatas", "documents"])

{'ids': ['8889fe21-079e-48cf-b2fe-cce2c44d36a4',
  'e931680c-d6a5-417a-a43e-8ebc8da419c1',
  '2c2dc62a-6d11-4608-bbf6-b458bedf6110',
  '3c64f402-1e59-45da-b265-da5cc47406a2',
  '71881230-f996-42bd-9142-207058c9bf1a'],
 'embeddings': array([[ 0.00994726,  0.06914339, -0.05147114, ..., -0.03543339,
          0.0128481 ,  0.0124829 ],
        [ 0.00127745,  0.0312985 , -0.02375377, ..., -0.00518358,
         -0.03280613,  0.02737715],
        [-0.10265917,  0.02650809,  0.02271502, ..., -0.03359748,
         -0.07984944, -0.01507708],
        [ 0.02123395, -0.02468547, -0.04494376, ..., -0.1099581 ,
          0.0057256 ,  0.09915379],
        [ 0.0187398 ,  0.04382846, -0.04304254, ..., -0.07801617,
         -0.07840686, -0.0030419 ]], shape=(5, 384)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the mo

In [17]:
vector_store.similarity_search("Which players are in Mumbai Indians?", k=2)

[Document(id='3c64f402-1e59-45da-b265-da5cc47406a2', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='e931680c-d6a5-417a-a43e-8ebc8da419c1', metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [18]:
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_documents(ids=["8889fe21-079e-48cf-b2fe-cce2c44d36a4"], documents=[updated_doc1])

In [19]:
vector_store.get(include=["embeddings", "metadatas", "documents"])

{'ids': ['8889fe21-079e-48cf-b2fe-cce2c44d36a4',
  'e931680c-d6a5-417a-a43e-8ebc8da419c1',
  '2c2dc62a-6d11-4608-bbf6-b458bedf6110',
  '3c64f402-1e59-45da-b265-da5cc47406a2',
  '71881230-f996-42bd-9142-207058c9bf1a'],
 'embeddings': array([[-0.00233745,  0.05902084, -0.04774046, ..., -0.07264046,
          0.00276784, -0.00344087],
        [ 0.00127745,  0.0312985 , -0.02375377, ..., -0.00518358,
         -0.03280613,  0.02737715],
        [-0.10265917,  0.02650809,  0.02271502, ..., -0.03359748,
         -0.07984944, -0.01507708],
        [ 0.02123395, -0.02468547, -0.04494376, ..., -0.1099581 ,
          0.0057256 ,  0.09915379],
        [ 0.0187398 ,  0.04382846, -0.04304254, ..., -0.07801617,
         -0.07840686, -0.0030419 ]], shape=(5, 384)),
 'documents': ["Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple ce